### Dimensión de Cuentas

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(CONCAT(:catalogo, '.dimensiones_comunes_', :alumno, '.dim_cuentas')) (
    id_dim_cuenta BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1) COMMENT 'Clave sustituta SCD2 que identifica un registro histórico único de la cuenta',
    id_cuenta BIGINT COMMENT 'Clave natural de negocio de la cuenta proveniente del sistema fuente',
    nombre_cuenta STRING COMMENT 'Nombre de la cuenta',
    correo_electronico STRING COMMENT 'Correo electrónico asociado a la cuenta',
    fecha_creacion TIMESTAMP COMMENT 'Fecha de creación de la cuenta en el sistema fuente',
    fecha_actualizacion TIMESTAMP COMMENT 'Última fecha de actualización de la cuenta en el sistema fuente',
    valido_desde DATE COMMENT 'Inicio de validez del registro SCD2',
    valido_hasta DATE COMMENT 'Fin de validez del registro SCD2',
    es_actual BOOLEAN COMMENT 'Indicador booleano de si el registro es la versión vigente',
    CONSTRAINT dim_cuentas_pk PRIMARY KEY(id_dim_cuenta)
)
USING DELTA
CLUSTER BY(id_dim_cuenta)
COMMENT 'Dimensión de cuentas tipo SCD2.';

**Sobre los constraints de PRIMARY KEY y FOREIGN KEY en Delta Lake:**

En Delta Lake, los constraints de PK/FK son **metadata informativa**, NO se validan automáticamente:

* **Documentación**: Declaran explícitamente las relaciones del modelo de datos
* **Herramientas de BI**: Permiten que herramientas de visualización entiendan relaciones y generen joins automáticamente
* **Catálogo**: Aparecen en Unity Catalog como metadata semántica
* **Optimización**: Algunos motores pueden usar esta info para optimizar queries
* ⚠️ **NO validan**: Delta NO rechaza duplicados en PKs ni valores inválidos en FKs

In [0]:
-- Insertar el registro -1 para cuentas desconocidas/no disponibles
-- Aunque id_dim_cuenta sea IDENTITY, podemos insertar valores explícitos porque fue creada con GENERATED BY DEFAULT AS IDENTITY, sin el BY DEFAULT no sería posible

INSERT INTO IDENTIFIER(CONCAT(:catalogo, '.dimensiones_comunes_', :alumno, '.dim_cuentas'))
    (id_dim_cuenta, id_cuenta, nombre_cuenta, correo_electronico, 
     fecha_creacion, fecha_actualizacion, valido_desde, valido_hasta, es_actual)
VALUES (
    -1,
    -1,
    'Desconocido',
    'N/A',
    TIMESTAMP '1900-01-01 00:00:00',
    TIMESTAMP '1900-01-01 00:00:00',
    DATE '1900-01-01',
    DATE '9999-12-31',
    TRUE
);

### Dimensión de Suscripciones


In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(CONCAT(:catalogo, '.dimensiones_comunes_', :alumno, '.dim_suscripciones')) (
    id_dim_suscripcion BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1) COMMENT 'Clave sustituta SCD2 para la suscripción',
    id_suscripcion BIGINT COMMENT 'Clave natural de la suscripción',
    nombre_suscripcion STRING COMMENT 'Nombre del plan de suscripción',
    max_contenidos_mensuales INT COMMENT 'Límite de contenidos permitidos por mes del plan',
    fecha_creacion TIMESTAMP COMMENT 'Fecha de creación del plan en el sistema fuente',
    fecha_actualizacion TIMESTAMP COMMENT 'Última fecha de actualización del plan en el sistema fuente',
    valido_desde DATE COMMENT 'Inicio de validez del registro SCD2',
    valido_hasta DATE COMMENT 'Fin de validez del registro SCD2',
    es_actual BOOLEAN COMMENT 'Indicador booleano de si el registro es la versión vigente',
    CONSTRAINT dim_suscripciones_pk PRIMARY KEY(id_dim_suscripcion)
)
USING DELTA
COMMENT 'Dimensión de suscripciones (SCD Tipo 2) construida a partir del snapshot de suscripciones. Mantiene la historia de cambios en atributos como nombre y límites del plan.';

In [0]:
-- Insertar el registro -1 para suscripciones desconocidas/no disponibles

INSERT INTO IDENTIFIER(CONCAT(:catalogo, '.dimensiones_comunes_', :alumno, '.dim_suscripciones'))
    (id_dim_suscripcion, id_suscripcion, nombre_suscripcion, max_contenidos_mensuales,
     fecha_creacion, fecha_actualizacion, valido_desde, valido_hasta, es_actual)
VALUES (
    -1,
    -1,
    'Desconocido',
    0,
    TIMESTAMP '1900-01-01 00:00:00',
    TIMESTAMP '1900-01-01 00:00:00',
    DATE '1900-01-01',
    DATE '9999-12-31',
    TRUE
);

### Dimensión de Funcionalidades Premium

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(CONCAT(:catalogo, '.dimensiones_comunes_', :alumno, '.dim_funcionalidades_premium')) (
    id_dim_funcionalidad BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1) COMMENT 'Clave sustituta SCD2 para la funcionalidad premium',
    id_funcionalidad BIGINT COMMENT 'Clave natural de la funcionalidad premium',
    nombre_funcionalidad STRING COMMENT 'Nombre descriptivo de la funcionalidad',
    descripcion STRING COMMENT 'Descripción detallada de la funcionalidad',
    precio_base DECIMAL(10,2) COMMENT 'Precio base de la funcionalidad',
    fecha_creacion TIMESTAMP COMMENT 'Fecha de creación de la funcionalidad en el sistema fuente',
    fecha_actualizacion TIMESTAMP COMMENT 'Última fecha de actualización de la funcionalidad en el sistema fuente',
    valido_desde DATE COMMENT 'Inicio de validez del registro SCD2',
    valido_hasta DATE COMMENT 'Fin de validez del registro SCD2',
    es_actual BOOLEAN COMMENT 'Indicador booleano de si el registro es la versión vigente',
    CONSTRAINT dim_funcionalidades_premium_pk PRIMARY KEY(id_dim_funcionalidad)
)
USING DELTA
COMMENT 'Dimensión de funcionalidades premium (SCD Tipo 2). Mantiene la historia de cambios en atributos como nombre, descripción y precio base.'

In [0]:
-- Insertar el registro -1 para funcionalidades premium desconocidas/no disponibles

INSERT INTO IDENTIFIER(CONCAT(:catalogo, '.dimensiones_comunes_', :alumno, '.dim_funcionalidades_premium'))
    (id_dim_funcionalidad, id_funcionalidad, nombre_funcionalidad, descripcion, precio_base,
     fecha_creacion, fecha_actualizacion, valido_desde, valido_hasta, es_actual)
VALUES (
    -1,
    -1,
    'Desconocido',
    'Funcionalidad no disponible o desconocida',
    0.00,
    TIMESTAMP '1900-01-01 00:00:00',
    TIMESTAMP '1900-01-01 00:00:00',
    DATE '1900-01-01',
    DATE '9999-12-31',
    TRUE
);

### Hecho de Suscripciones Cuentas

In [0]:
%python
# Get parameter values
catalogo = dbutils.widgets.get('catalogo')
alumno = dbutils.widgets.get('alumno')

# Create fact table
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.analisis_suscripciones_{alumno}.fact_suscripciones_cuentas (
    id_dim_cuenta BIGINT COMMENT 'Clave de dimensión de cuenta válida para fecha_inicio del período',
    id_dim_suscripcion BIGINT COMMENT 'Clave de dimensión de la suscripción válida para fecha_inicio del período',
    id_dim_suscripcion_anterior BIGINT COMMENT 'Clave de dimensión de la suscripción anterior válida al inicio del período (si existe)',
    fecha_inicio DATE COMMENT 'Fecha de inicio del período de la suscripción en la cuenta',
    fecha_fin DATE COMMENT 'Fecha de fin del período (nulo si el período está aún vigente)',
    duracion_dias INT COMMENT 'Duración del período en días (usa current_date cuando no hay fecha_fin)',
    tipo_cambio STRING COMMENT 'Tipo de cambio del plan respecto al período anterior (Upgrade/Downgrade/Misma/Nueva)',
    fecha_churn DATE COMMENT 'Fecha de churn cuando hay fecha_fin y no existe suscripción siguiente',
    dias_desde_cambio_anterior INT COMMENT 'Días transcurridos entre el inicio del período anterior y el churn del período actual (solo cuando aplica churn)',
    dias_desde_inicio_cuenta INT COMMENT 'Días transcurridos entre la primera suscripción de la cuenta y el churn del período actual (solo cuando aplica churn)',
    CONSTRAINT fact_suscripciones_cuentas_fk_cuenta FOREIGN KEY (id_dim_cuenta) REFERENCES {catalogo}.dimensiones_comunes_{alumno}.dim_cuentas(id_dim_cuenta),
    CONSTRAINT fact_suscripciones_cuentas_fk_suscripcion FOREIGN KEY (id_dim_suscripcion) REFERENCES {catalogo}.dimensiones_comunes_{alumno}.dim_suscripciones(id_dim_suscripcion)
)
USING DELTA
CLUSTER BY (id_dim_cuenta, id_dim_suscripcion, fecha_inicio)
COMMENT 'Hecho que modela los períodos de suscripción por cuenta y los eventos de cambio entre planes. Se deriva de cuentas_suscripciones y determina para cada período el tipo de cambio respecto al plan anterior.'
""")

### Hecho de Funcionalidades Premium Cuentas

In [0]:
%python
# Get parameter values
catalogo = dbutils.widgets.get('catalogo')
alumno = dbutils.widgets.get('alumno')

# Create fact table
spark.sql(f"""
CREATE OR REPLACE TABLE {catalogo}.analisis_suscripciones_{alumno}.fact_funcionalidades_premium_cuentas (
    id_dim_cuenta BIGINT COMMENT 'Clave de dimensión de cuenta válida para la fecha de compra',
    id_dim_funcionalidad BIGINT COMMENT 'Clave de dimensión de la funcionalidad premium válida para la fecha de compra',
    fecha_compra DATE COMMENT 'Fecha en que se adquirió la funcionalidad',
    monto_pagado DECIMAL(10,2) COMMENT 'Monto pagado por la funcionalidad',
    CONSTRAINT fact_funcionalidades_premium_cuentas_fk_cuenta FOREIGN KEY (id_dim_cuenta) REFERENCES {catalogo}.dimensiones_comunes_{alumno}.dim_cuentas(id_dim_cuenta),
    CONSTRAINT fact_funcionalidades_premium_cuentas_fk_funcionalidad FOREIGN KEY (id_dim_funcionalidad) REFERENCES {catalogo}.dimensiones_comunes_{alumno}.dim_funcionalidades_premium(id_dim_funcionalidad)
)
USING DELTA
CLUSTER BY (fecha_compra)
COMMENT 'Hecho que registra las compras de funcionalidades premium por cuenta. Incluye métricas de precio y descuento aplicado.'
""")